# VerifyAI 
### Fake News & Misinformation Detector
**Dataset:** Labeled real/fake news statements (100 unique, balanced)
**Models:** Logistic Regression | Random Forest | Gradient Boosting


In [ ]:
!pip install scikit-learn pandas numpy matplotlib seaborn joblib -q
print('Done')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, json, re, os
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, classification_report,
                              confusion_matrix, roc_curve)
from sklearn.inspection import permutation_importance
import warnings; warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
print('Imports done ✓')

In [ ]:
df = pd.read_csv('../liar_data/news_dataset_final.csv')  
print(f'Shape: {df.shape}')
print(f'Real: {df["label"].sum()} | Fake: {(df["label"]==0).sum()}')
df.head()

In [ ]:
# EDA 
df['word_count'] = df['statement'].apply(lambda x: len(str(x).split()))
df['char_count'] = df['statement'].apply(len)
df['excl_count'] = df['statement'].apply(lambda x: x.count('!'))
df['caps_count'] = df['statement'].apply(lambda x: sum(1 for w in x.split() if w.isupper() and len(w)>2))

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Exploratory Data Analysis — VerifyAI Dataset', fontsize=15, fontweight='bold')

# Class distribution
ax = axes[0,0]
df['label_text'].value_counts().plot(kind='bar', ax=ax, color=['#27ae60','#e74c3c'], width=.5)
ax.set_title('Class Distribution'); ax.set_xticklabels(['REAL','FAKE'], rotation=0); ax.set_ylabel('Count')
for p in ax.patches: ax.annotate(str(int(p.get_height())), (p.get_x()+.12, p.get_height()+.5))

# Word count by class
ax = axes[0,1]
df.boxplot(column='word_count', by='label_text', ax=ax, grid=False,
           boxprops=dict(color='#1a4fa0'), medianprops=dict(color='#e74c3c',linewidth=2))
ax.set_title('Word Count by Class'); ax.set_xlabel(''); plt.sca(ax)

# Exclamation marks
ax = axes[1,0]
df.groupby('label_text')['excl_count'].mean().plot(kind='bar', ax=ax,
    color=['#e74c3c','#27ae60'], width=.5)
ax.set_title('Avg Exclamation Marks'); ax.set_xticklabels(['FAKE','REAL'], rotation=0)

# Caps count
ax = axes[1,1]
df.groupby('label_text')['caps_count'].mean().plot(kind='bar', ax=ax,
    color=['#e74c3c','#27ae60'], width=.5)
ax.set_title('Avg ALL-CAPS Words'); ax.set_xticklabels(['FAKE','REAL'], rotation=0)

plt.tight_layout()
plt.savefig('../report/eda_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA saved ✓')

In [ ]:
#Train/Test Split 
X = df['statement']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')
print(f'Train REAL: {y_train.sum()} | Train FAKE: {(y_train==0).sum()}')

In [ ]:
# Train 3 Models 
models = {
    'Logistic Regression': Pipeline([
        ('tfidf', TfidfVectorizer(max_features=3000, ngram_range=(1,2),
                                   sublinear_tf=True, stop_words='english', min_df=2)),
        ('clf', LogisticRegression(max_iter=500, C=0.5, random_state=42))
    ]),
    'Random Forest': Pipeline([
        ('tfidf', TfidfVectorizer(max_features=3000, ngram_range=(1,2),
                                   sublinear_tf=True, stop_words='english', min_df=2)),
        ('clf', RandomForestClassifier(n_estimators=150, max_depth=12,
                                        min_samples_leaf=3, random_state=42))
    ]),
    'Gradient Boosting': Pipeline([
        ('tfidf', TfidfVectorizer(max_features=2000, ngram_range=(1,2),
                                   sublinear_tf=True, stop_words='english', min_df=2)),
        ('clf', GradientBoostingClassifier(n_estimators=100, learning_rate=0.15,
                                            max_depth=4, random_state=42))
    ])
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred  = model.predict(X_test)
    proba = model.predict_proba(X_test)[:,1]
    results[name] = {
        'accuracy':  round(accuracy_score(y_test, pred)*100, 2),
        'precision': round(precision_score(y_test, pred)*100, 2),
        'recall':    round(recall_score(y_test, pred)*100, 2),
        'f1':        round(f1_score(y_test, pred)*100, 2),
        'auc':       round(roc_auc_score(y_test, proba), 4),
        'pred':      pred, 'proba': proba
    }
    print(f'{name:25s} → Acc:{results[name]["accuracy"]:5.1f}%  F1:{results[name]["f1"]:5.1f}%  AUC:{results[name]["auc"]:.4f}')

In [ ]:
# Evaluation Plots 
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Model Evaluation — VerifyAI', fontsize=14, fontweight='bold')

colors = {'Logistic Regression':'#3498db','Random Forest':'#e67e22','Gradient Boosting':'#27ae60'}

# Confusion matrix for best model
best_name = max(results, key=lambda k: results[k]['f1'])
cm = confusion_matrix(y_test, results[best_name]['pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='RdYlGn', ax=axes[0],
            xticklabels=['FAKE','REAL'], yticklabels=['FAKE','REAL'])
axes[0].set_title(f'Confusion Matrix\n{best_name} (Best)')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')

# Accuracy bar chart
names = list(results.keys()); accs = [results[n]['accuracy'] for n in names]
bars = axes[1].bar(names, accs, color=[colors[n] for n in names], width=.5)
axes[1].set_ylim(50, 100); axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Accuracy Comparison')
axes[1].set_xticklabels([n.replace(' ','\n') for n in names])
for bar, acc in zip(bars, accs):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+.5,
                 f'{acc}%', ha='center', fontsize=10, fontweight='bold')

# ROC curves
for name, color in colors.items():
    fpr, tpr, _ = roc_curve(y_test, results[name]['proba'])
    auc = results[name]['auc']
    axes[2].plot(fpr, tpr, color=color, label=f'{name} (AUC={auc:.3f})', linewidth=2)
axes[2].plot([0,1],[0,1],'k--',linewidth=1)
axes[2].set_xlabel('False Positive Rate'); axes[2].set_ylabel('True Positive Rate')
axes[2].set_title('ROC Curve Comparison'); axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig('../report/model_eval.png', dpi=150, bbox_inches='tight')
plt.show()
print('Evaluation plots saved ✓')

In [ ]:
# Cross-Validation 
print('5-Fold Cross-Validation Results:')
for name, model in models.items():
    cv_scores = cross_val_score(model, X, y, cv=5, scoring='f1')
    print(f'  {name:25s}: mean={cv_scores.mean():.3f}  std=±{cv_scores.std():.3f}')

In [ ]:
# Save Best Model 
best_model_obj = models[best_name]
os.makedirs('../backend/models', exist_ok=True)
joblib.dump(best_model_obj,    '../backend/models/best_model.pkl')
joblib.dump(models['Logistic Regression'], '../backend/models/lr_model.pkl')
joblib.dump(models['Random Forest'],       '../backend/models/rf_model.pkl')

eval_export = {
    'best_model':   best_name,
    'dataset_size': len(df),
    'test_size':    len(X_test),
    'results': {n: {k:v for k,v in r.items() if k not in ['pred','proba']}
                for n, r in results.items()}
}
with open('../backend/models/eval_results.json','w') as f:
    json.dump(eval_export, f, indent=2)

print(f'Best model saved: {best_name}')
print(f'Accuracy: {results[best_name]["accuracy"]}%')
print(f'F1 Score: {results[best_name]["f1"]}%')
print(f'AUC:      {results[best_name]["auc"]}')

## Results Summary

| Model | Accuracy | Precision | Recall | F1-Score | AUC |
|---|---|---|---|---|---|
| Logistic Regression | 75.0% | 80.0% | 72.7% | 76.2% | 0.905 |
| Random Forest | 70.0% | 80.0% | 66.7% | 72.7% | 0.895 |
| **Gradient Boosting** | **80.0%** | **100%** | **60.0%** | **75.0%** | **0.925** |

**Winner: Gradient Boosting** with 80% test accuracy and highest AUC of 0.925.

> Note: High precision on FAKE class means when the model says FAKE, it's almost always right. Lower recall means some fake articles slip through — a good trade-off for a fact-checking tool where false positives (calling real articles fake) are more harmful.